# 03b · BIOSCAN-5M, a second image source with real undescribed species

A second data source with different imaging conditions, and novel species that are real rather than
hold-outs simulated from iNaturalist. BIOSCAN-5M provides both:

- **Different imaging:** insect specimens from field traps, photographed under a microscope in a lab.
- **Real novel species:** its open-world partitions contain species known only by placeholder names,
  distinguished by DNA barcode. These are genuinely undescribed taxa, not hold-outs we created.

| Partition | What it contains | Role here |
|---|---|---|
| `train` | species with scientific names | known reference |
| `key_unseen` / `val_unseen` / `test_unseen` | undescribed species whose **genus** is named | real novel species |
| `other_heldout` | undescribed species, often with an undescribed genus too | real novel species, coarser attachment |

The ground truth for every novel species is again **computed** from its deepest ancestor in the known set.

**Download:** about 6 GB (metadata, plus the training and evaluation image archives at 256 px). The
4.7-million-image pretrain partition is never downloaded. Everything goes to `files/bioscan/` and
`files/v2_bioscan/`; nothing else is touched.

**Licence:** BIOSCAN-5M images and metadata are CC BY 3.0. Cite Gharaee et al., NeurIPS 2024 (Datasets and
Benchmarks).

## Step 1 — Configuration

In [ ]:
# CELL: setup
import json, os, sys, time, random, shutil, zipfile, hashlib, urllib.request, urllib.error
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch

BASE_DIR     = Path(os.environ.get("HNSD_BASE_DIR", r"C:\hnsd\files"))   # data root; set HNSD_BASE_DIR or edit
BIOSCAN_ROOT = BASE_DIR / "bioscan"                    # downloads and extracted images
V2_DIR       = BASE_DIR / "v2_bioscan"                 # everything this notebook produces
SHARD_DIR    = V2_DIR / "feature_shards"
INAT_FUSION  = BASE_DIR / "v2" / "fusion_main" / "best.pt"   # trained in Step 3, applied here unchanged

SEED                      = 42
IMGS_PER_SPECIES          = 10     # cap per species (BIOSCAN species typically have few images)
MIN_IMAGES                = 3      # species with fewer images are dropped
OTHER_HELDOUT_MAX_SPECIES = 2000   # random sample of other_heldout species
N_VAL_SPECIES             = 120

SIGLIP2_ID = "google/siglip2-so400m-patch16-naflex"
DINOV3_ID  = "facebook/dinov3-vitl16-pretrain-lvd1689m"
DINOV2_ID  = "facebook/dinov2-large"
HF_TOKEN   = None
BATCH, IO_WORKERS = 48, 8
SHARD_SPECIES = 200                # ~2,000 images per resumable shard

RANKS  = ["kingdom", "phylum", "class", "order", "family", "genus"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
for d in (BIOSCAN_ROOT, V2_DIR, SHARD_DIR):
    d.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(BASE_DIR).free / 1e9
print(f"device {DEVICE} | free disk {free_gb:.1f} GB (needs ~13 GB: 6 GB archives + 6 GB extracted)")
if free_gb < 15:
    print("WARNING: under 15 GB free.")

## Step 2 — Download and verify

Three files, fetched from the official Zenodo record with a Hugging Face mirror for the metadata. Each is
checked against the MD5 published in the official loader. Downloads resume where possible, and a file is only
extracted once.

In [ ]:
# CELL: download
FILES = {
    "metadata": (["https://zenodo.org/records/11973457/files/BIOSCAN_5M_Insect_Dataset_metadata_MultiTypes.zip",
                  "https://huggingface.co/datasets/bioscan-ml/BIOSCAN-5M/resolve/eeefb30/BIOSCAN_5M_Insect_Dataset_metadata_MultiTypes.zip"],
                 "ac381b69fafdbaedc2f9cfb89e3571f7"),
    "train":    (["https://zenodo.org/records/11973457/files/BIOSCAN_5M_cropped_256_train.zip"],
                 "3f170db5d95610644883dafc73389049"),
    "eval":     (["https://zenodo.org/records/11973457/files/BIOSCAN_5M_cropped_256_eval.zip"],
                 "16a997f72a8cd08cbcf7becafe2dda50"),
}

def md5sum(path, chunk=1 << 22):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def download(url, dest):
    have = dest.stat().st_size if dest.exists() else 0
    req = urllib.request.Request(url, headers={"Range": f"bytes={have}-"} if have else {})
    try:
        r = urllib.request.urlopen(req, timeout=120)
    except urllib.error.HTTPError as e:
        if e.code == 416:          # range beyond end: file already complete
            return
        raise
    with r:
        if have and getattr(r, "status", 200) != 206:
            have = 0               # server ignored the range: start over
        total = have + int(r.headers.get("Content-Length") or 0)
        done, t0, last = have, time.time(), 0
        with open(dest, "ab" if have else "wb") as f:
            while True:
                b = r.read(1 << 20)
                if not b:
                    break
                f.write(b); done += len(b)
                if done - last > 200e6:
                    last = done
                    print(f"  {dest.name}: {done/1e9:.2f}/{total/1e9:.2f} GB ({done/1e6/max(time.time()-t0,1):.1f} MB/s)")

def fetch(urls, want):
    dest = BIOSCAN_ROOT / urls[0].rsplit("/", 1)[1]
    if dest.exists() and md5sum(dest) == want:
        print(f"{dest.name}: present and verified"); return dest
    for attempt, url in enumerate(urls * 2):
        print(f"{dest.name}: downloading from {url.split('/')[2]} ...")
        try:
            download(url, dest)
        except Exception as e:
            print(f"  failed: {e}"); continue
        if md5sum(dest) == want:
            print(f"{dest.name}: verified"); return dest
        print("  checksum mismatch, restarting from scratch"); dest.unlink()
    raise RuntimeError(f"could not download {dest.name}")

def extract(zp):
    marker = BIOSCAN_ROOT / f".extracted_{zp.stem}"
    if marker.exists():
        print(f"{zp.name}: already extracted"); return
    print(f"{zp.name}: extracting ...")
    with zipfile.ZipFile(zp) as z:
        z.extractall(BIOSCAN_ROOT)
    marker.write_text("ok")

for name, (urls, want) in FILES.items():
    extract(fetch(urls, want))

CSV = next(BIOSCAN_ROOT.rglob("BIOSCAN_5M_Insect_Dataset_metadata.csv"), None)
guess = BIOSCAN_ROOT / "bioscan5m" / "images" / "cropped_256"
IMG_ROOT = guess if (guess / "train").exists() else next(BIOSCAN_ROOT.rglob("AACTA1026-20.jpg")).parents[2]
assert CSV is not None, "metadata CSV not found after extraction"
print("metadata:", CSV)
print("images  :", IMG_ROOT)

## Step 3 — Load the metadata

Only the needed columns and partitions are read; DNA sequences are skipped to save memory. Image paths are
rebuilt exactly as the official loader builds them: `<split>/<chunk>/<processid>.jpg`.

In [ ]:
# CELL: metadata
USE = ["processid", "chunk", "phylum", "class", "order", "family", "subfamily", "genus", "species", "dna_bin", "split"]
NEED = ["train", "key_unseen", "val_unseen", "test_unseen", "other_heldout"]
t0 = time.time()
META = pd.read_csv(CSV, usecols=USE, dtype=str)
META = META[META["split"].isin(NEED)].copy()
chunk = META["chunk"].fillna("")
META["image_path"] = META["split"] + "/" + chunk.map(lambda c: f"{c}/" if c else "") + META["processid"] + ".jpg"
print(f"loaded in {time.time()-t0:.0f}s: {len(META)} rows")
print(META["split"].value_counts().to_string())

## Step 4 — Design the benchmark and compute the ground truth

- **Known:** every `train` species with at least `MIN_IMAGES` images. Using all of them avoids any
  selection choice that could favour particular novel species.
- **Novel:** all unseen species (their images pooled across `key_unseen`, `val_unseen` and `test_unseen`, which
  share the same species), plus a random sample of `other_heldout` species.
- **Ground truth:** for each novel species, the deepest rank whose lineage prefix exists among the known species.
  Missing labels are given a token unique to that species, so they can never match anything.

Kingdom is set to Animalia for every specimen, since BIOSCAN contains only arthropods.

In [ ]:
# CELL: design
TAX = ["phylum", "class", "order", "family", "genus"]
M = META[META["species"].notna()].copy()
M["group"] = np.where(M["split"] == "train", "known", np.where(M["split"] == "other_heldout", "other", "unseen"))
multi = M.groupby("species")["group"].nunique()
multi = set(multi[multi > 1].index)
if multi:
    print(f"dropping {len(multi)} species found in more than one group")
    M = M[~M["species"].isin(multi)]

LIN   = M.groupby("species")[TAX + ["subfamily"]].first()      # first non-null label per species
N_IMG = M.groupby("species").size()
GRP   = M.groupby("species")["group"].first()
ok = N_IMG[N_IMG >= MIN_IMAGES].index
known_sp  = sorted(s for s in ok if GRP[s] == "known")
unseen_sp = sorted(s for s in ok if GRP[s] == "unseen")
other_all = sorted(s for s in ok if GRP[s] == "other")
rng = np.random.default_rng(SEED)
other_sp = sorted(rng.choice(other_all, size=min(OTHER_HELDOUT_MAX_SPECIES, len(other_all)), replace=False).tolist())
CHOSEN = sorted(known_sp + unseen_sp + other_sp)
CAT_ID = {s: i for i, s in enumerate(CHOSEN)}

def lineage(sp):
    rec = {"kingdom": "Animalia"}
    for r in TAX:
        v = LIN.loc[sp, r]
        rec[r] = v if isinstance(v, str) and v.strip() else f"__missing__{r}__{sp}"
    return rec

def prefix(s, r):
    return tuple(s[x] for x in RANKS[:RANKS.index(r) + 1])

HOLD = {"known": "none", "unseen": "bioscan_unseen", "other": "bioscan_other_heldout"}
RECS = {}
for sp in CHOSEN:
    sub = LIN.loc[sp, "subfamily"]
    RECS[sp] = {"category_id": CAT_ID[sp], "name": sp,
                "supercategory": LIN.loc[sp, "order"] if isinstance(LIN.loc[sp, "order"], str) else "unknown",
                **lineage(sp), "subfamily": sub if isinstance(sub, str) else None,
                "holdout": HOLD[GRP[sp]], "num_images": int(min(N_IMG[sp], IMGS_PER_SPECIES))}

known_prefixes = {r: {prefix(RECS[s], r) for s in known_sp} for r in RANKS}
for sp in CHOSEN:
    rec = RECS[sp]
    if rec["holdout"] == "none":
        rec.update(role="known", true_rank="species", true_parent=sp)
        continue
    rank, parent = "none", None
    for r in RANKS:
        if prefix(rec, r) in known_prefixes[r]:
            rank, parent = r, rec[r]
        else:
            break
    rec.update(true_rank=rank, true_parent=parent,
               role="novel_known_branch" if rank in ("order", "family", "genus") else "novel_unseen_branch")

SPLIT = [RECS[s] for s in CHOSEN]
ROLE = {r["category_id"]: r["role"] for r in SPLIT}
json.dump(SPLIT, open(V2_DIR / "bioscan_split.json", "w"), indent=1)

print("ROLES:", dict(Counter(r["role"] for r in SPLIT)))
cols = ["genus", "family", "order", "class", "phylum", "kingdom", "none"]
print(f"{'source':<24}" + "".join(f"{c:>9}" for c in cols))
for h in ["bioscan_unseen", "bioscan_other_heldout"]:
    c = Counter(r["true_rank"] for r in SPLIT if r["holdout"] == h)
    print(f"{h:<24}" + "".join(f"{c.get(x, 0):>9}" for x in cols))
print(f"\nknown species {len(known_sp)} | unseen {len(unseen_sp)} | other_heldout {len(other_sp)} of {len(other_all)}")
print("top orders among known:", dict(Counter(RECS[s]["order"] for s in known_sp).most_common(6)))

## Step 5 — Fix the image list

Up to `IMGS_PER_SPECIES` images per species, drawn with a per-species seed so the selection is reproducible.
Any image missing on disk is dropped and reported, and image counts in the split file are updated.

In [ ]:
# CELL: items
rows = M[M["species"].isin(set(CHOSEN))]
ITEMS = []
for sp, g in rows.groupby("species"):
    paths = sorted(g["image_path"].tolist())
    if len(paths) > IMGS_PER_SPECIES:
        paths = sorted(np.random.default_rng(SEED + CAT_ID[sp]).choice(paths, IMGS_PER_SPECIES, replace=False).tolist())
    ITEMS += [(p, CAT_ID[sp]) for p in paths]
ITEMS.sort(key=lambda t: (t[1], t[0]))
gone = [p for p, _ in ITEMS if not (IMG_ROOT / p).exists()]
if gone:
    print(f"WARNING: {len(gone)} images listed in the metadata are missing on disk, e.g. {gone[:3]}")
    ITEMS = [t for t in ITEMS if (IMG_ROOT / t[0]).exists()]
cnt = Counter(c for _, c in ITEMS)
for r in SPLIT:
    r["num_images"] = cnt.get(r["category_id"], 0)
json.dump(SPLIT, open(V2_DIR / "bioscan_split.json", "w"), indent=1)

by_cat = defaultdict(list)
for it in ITEMS:
    by_cat[it[1]].append(it)
order = sorted(by_cat)
SHARDS = [[it for c in order[i:i + SHARD_SPECIES] for it in by_cat[c]] for i in range(0, len(order), SHARD_SPECIES)]
json.dump(ITEMS, open(V2_DIR / "items.json", "w"))
print(f"images: {len(ITEMS)} | species: {len(order)} | shards: {len(SHARDS)}")

## Steps 6–9 — Extractors, throughput test, extraction, merge

Identical to Step 3: same encoders, preprocessing, pooling and resumable sharding.

In [ ]:
# CELL: extract_defs
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

BACKBONES = {"siglip2": SIGLIP2_ID, "dinov3": DINOV3_ID, "dinov2": DINOV2_ID}

def load_backbone(name):
    from transformers import AutoModel, AutoProcessor, AutoImageProcessor
    mid = BACKBONES[name]
    if name == "siglip2":
        proc  = AutoProcessor.from_pretrained(mid)
        model = AutoModel.from_pretrained(mid)
    else:
        tok = (HF_TOKEN or True) if name == "dinov3" else None
        proc  = AutoImageProcessor.from_pretrained(mid, token=tok)
        model = AutoModel.from_pretrained(mid, token=tok)
    model = model.to(DEVICE).eval()
    for p in model.parameters():
        p.requires_grad = False
    return model, proc

@torch.no_grad()
def embed_images(name, model, proc, imgs):
    inp = proc(images=imgs, return_tensors="pt").to(DEVICE)
    if name == "siglip2":
        out = model.get_image_features(**inp)
        emb = out if torch.is_tensor(out) else out.pooler_output
    else:
        out = model(**inp)
        emb = out.pooler_output if getattr(out, "pooler_output", None) is not None else out.last_hidden_state[:, 0]
    emb = torch.nn.functional.normalize(emb.float(), dim=-1)
    return emb.cpu().numpy().astype(np.float32)

def load_rgb(rel):
    with Image.open(IMG_ROOT / rel) as im:
        return im.convert("RGB")

def shard_path(name, i):
    return SHARD_DIR / name / f"shard_{i:04d}.npz"

def run_backbone(name, batch=BATCH):
    (SHARD_DIR / name).mkdir(parents=True, exist_ok=True)
    todo = [i for i in range(len(SHARDS)) if not shard_path(name, i).exists()]
    if not todo:
        print(f"[{name}] all {len(SHARDS)} shards already done"); return
    print(f"[{name}] {len(todo)} of {len(SHARDS)} shards to do")
    model, proc = load_backbone(name)
    io = ThreadPoolExecutor(max_workers=IO_WORKERS)
    t0, n_img = time.time(), 0
    try:
        for k, i in enumerate(todo):
            items, feats, j = SHARDS[i], [], 0
            while j < len(items):
                chunk = items[j:j + batch]
                imgs = list(io.map(load_rgb, [r for r, _ in chunk]))
                try:
                    feats.append(embed_images(name, model, proc, imgs))
                    j += len(chunk)
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache()
                    batch = max(1, batch // 2)
                    print(f"  out of memory -> batch {batch}")
            tmp = shard_path(name, i).with_suffix(".tmp.npz")
            np.savez(tmp, X=np.concatenate(feats),
                     cat_ids=np.array([c for _, c in items], dtype=np.int64),
                     relpaths=np.array([r for r, _ in items]))
            os.replace(tmp, shard_path(name, i))     # atomic: a half-written shard never counts as done
            n_img += len(items)
            rate = n_img / max(time.time() - t0, 1e-6)
            left = sum(len(SHARDS[x]) for x in todo[k + 1:]) / max(rate, 1e-6)
            print(f"  shard {i+1}/{len(SHARDS)}  {rate:.0f} img/s  ~{left/60:.0f} min left for {name}")
    finally:
        io.shutdown()
        del model
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

print("extractors ready:", list(BACKBONES))

In [ ]:
# CELL: throughput
TEST_N = min(512, len(ITEMS))
model, proc = load_backbone("siglip2")
rels = [r for r, _ in ITEMS[:TEST_N]]
b = BATCH
with ThreadPoolExecutor(max_workers=IO_WORKERS) as io:
    embed_images("siglip2", model, proc, list(io.map(load_rgb, rels[:8])))      # warm-up
    t0, j = time.time(), 0
    while j < TEST_N:
        try:
            embed_images("siglip2", model, proc, list(io.map(load_rgb, rels[j:j + b])))
            j += b
        except torch.cuda.OutOfMemoryError:
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
            b = max(1, b // 2)
            print(f"out of memory -> batch {b}")
    dt = time.time() - t0
BATCH = b                                    # the extraction run starts from a batch size that fits
del model
if DEVICE == "cuda":
    torch.cuda.empty_cache()
ips = TEST_N / dt
print(f"SigLIP 2: {ips:.0f} images/s at batch {BATCH}")
print(f"rough estimate for all three backbones on {len(ITEMS)} images: ~{3*len(ITEMS)/ips/60:.0f} min")
print("(DINO models are usually a little faster than SigLIP 2, so this tends to overestimate.)")

In [ ]:
# CELL: extract_run
print(f"starting batch size: {BATCH}")
for name in BACKBONES:
    run_backbone(name, batch=BATCH)       # BATCH comes from the throughput test if it ran
print("all backbones done")

In [ ]:
# CELL: merge
def merge(name):
    parts = [np.load(shard_path(name, i)) for i in range(len(SHARDS))]
    X   = np.concatenate([p["X"] for p in parts])
    cid = np.concatenate([p["cat_ids"] for p in parts])
    rel = np.concatenate([p["relpaths"] for p in parts])
    np.savez(V2_DIR / f"features_{name}_v2.npz", X=X, cat_ids=cid, relpaths=rel)
    return X, cid, rel

FEATS = {}
for name in BACKBONES:
    FEATS[name] = merge(name)
    print(f"{name:<8} {FEATS[name][0].shape}")

ref_cid, ref_rel = FEATS["siglip2"][1], FEATS["siglip2"][2]
for name in BACKBONES:
    assert np.array_equal(FEATS[name][1], ref_cid), f"{name}: species order differs"
    assert np.array_equal(FEATS[name][2], ref_rel), f"{name}: image order differs"
    norms = np.linalg.norm(FEATS[name][0], axis=1)
    assert np.allclose(norms, 1, atol=1e-3), f"{name}: features not normalised"
assert len(ref_rel) == len(ITEMS)
CAT, REL = ref_cid, ref_rel
print("aligned: all three backbones share the same", len(CAT), "rows")

## Step 10 — Fusion module

The same architecture and loss as Step 3.

In [ ]:
# CELL: fusion_defs
import torch.nn as nn
import torch.nn.functional as F
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score

class TriFusion(nn.Module):
    def __init__(self, d_s2, d_d3, d_d2, d_model=256, n_heads=4, out_dim=512, p=0.4):
        super().__init__()
        self.p_s2 = nn.Linear(d_s2, d_model); self.p_d3 = nn.Linear(d_d3, d_model); self.p_d2 = nn.Linear(d_d2, d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=p, batch_first=True)
        self.norm = nn.LayerNorm(d_model); self.drop = nn.Dropout(p)
        self.mlp  = nn.Sequential(nn.Linear(3 * d_model, d_model), nn.GELU(), nn.Dropout(p), nn.Linear(d_model, out_dim))
        self.skip = nn.Linear(d_s2 + d_d3 + d_d2, out_dim)
        self.alpha = nn.Parameter(torch.tensor(0.1))
    def forward(self, a, b, c):
        t = torch.cat([self.p_s2(a).unsqueeze(1), self.p_d3(b).unsqueeze(1), self.p_d2(c).unsqueeze(1)], dim=1)
        ctx, _ = self.attn(t, t, t)
        ctx = self.norm(t + self.drop(ctx))
        adjust = self.mlp(ctx.reshape(ctx.size(0), -1))
        anchor = self.skip(torch.cat([a, b, c], dim=-1))
        return F.normalize(anchor + self.alpha * adjust, dim=-1)

def supcon_loss(z, labels, temp=0.1):
    sim = z @ z.T / temp
    eye = torch.eye(z.size(0), dtype=torch.bool, device=z.device)
    sim = sim.masked_fill(eye, -9e15)
    same = (labels[:, None] == labels[None, :]) & ~eye
    logp = sim - torch.logsumexp(sim, dim=1, keepdim=True)
    pc = same.sum(1); v = pc > 0
    if not v.any():
        return z.sum() * 0.0
    return (-(logp * same).sum(1)[v] / pc[v].clamp(min=1)).mean()

XS_T = torch.tensor(FEATS["siglip2"][0], device=DEVICE)
XD3_T = torch.tensor(FEATS["dinov3"][0], device=DEVICE)
XD2_T = torch.tensor(FEATS["dinov2"][0], device=DEVICE)
CAT_T = torch.tensor(CAT.astype(np.int64), device=DEVICE)

def fused_features(model, idx, bs=4096):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(idx), bs):
            j = torch.as_tensor(idx[i:i + bs], device=DEVICE)
            out.append(model(XS_T[j], XD3_T[j], XD2_T[j]).cpu().numpy())
    return np.concatenate(out).astype(np.float32)

def train_fusion(tag, train_species, val_species, out_dir, epochs=80, batch=256, lr=5e-5, wd=1e-2, patience=40):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    ck_best, ck_last, log_p = out_dir / "best.pt", out_dir / "latest.pt", out_dir / "log.json"
    tr_idx = np.where(np.isin(CAT, sorted(train_species)))[0]
    va_idx = np.where(np.isin(CAT, sorted(val_species)))[0]
    k_val  = len(np.unique(CAT[va_idx]))
    model = TriFusion(XS_T.shape[1], XD3_T.shape[1], XD2_T.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    start, best, bad, hist = 0, -1.0, 0, []
    if ck_last.exists():
        ck = torch.load(ck_last, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        start, best, bad = ck["epoch"] + 1, ck["best"], ck["bad"]
        hist = json.load(open(log_p)) if log_p.exists() else []
        print(f"[{tag}] resuming at epoch {start + 1}, best val NMI {best:.4f}")
    for ep in range(start, epochs):
        if bad >= patience:
            break
        model.train()
        order = np.random.default_rng(SEED + ep).permutation(tr_idx)
        tot, nb = 0.0, 0
        for i in range(0, len(order), batch):
            j = torch.as_tensor(order[i:i + batch], device=DEVICE)
            loss = supcon_loss(model(XS_T[j], XD3_T[j], XD2_T[j]), CAT_T[j])
            opt.zero_grad(); loss.backward(); opt.step()
            tot += float(loss); nb += 1
        f = fused_features(model, va_idx)
        nmi = normalized_mutual_info_score(CAT[va_idx], KMeans(k_val, random_state=SEED, n_init=3).fit_predict(f))
        improved = nmi > best
        if improved:
            best, bad = nmi, 0
            torch.save({"model": model.state_dict(), "best": best}, ck_best)
        else:
            bad += 1
        hist.append({"epoch": ep + 1, "loss": tot / max(nb, 1), "val_nmi": nmi, "best": best})
        json.dump(hist, open(log_p, "w"), indent=1)
        torch.save({"epoch": ep, "best": best, "bad": bad, "model": model.state_dict(), "opt": opt.state_dict()}, ck_last)
        print(f"[{tag}] ep {ep+1:3d}  loss {tot/max(nb,1):.4f}  val NMI {nmi:.4f}" + ("  *best*" if improved else ""))
    model.load_state_dict(torch.load(ck_best, map_location=DEVICE, weights_only=False)["model"])
    print(f"[{tag}] best val NMI {best:.4f}")
    return fused_features(model, np.arange(len(CAT))), model

def pick_val(known_species, n_val, seed=SEED):
    ks = np.array(sorted(known_species))
    ks = np.random.default_rng(seed).permutation(ks)
    n_val = min(n_val, len(ks) // 5)
    return set(ks[n_val:].tolist()), set(ks[:n_val].tolist())

print("fusion ready on", DEVICE, "| rows:", len(CAT))

## Step 11 — Train a BIOSCAN fusion with species-balanced batches

**One procedural change, to report in the paper.** BIOSCAN species have about 10 images each, against 50 in
iNaturalist. With plain random batches almost no image would have a same-species partner in its batch, and the
contrastive loss would carry almost no signal. Each batch here is therefore 64 species × 4 images, so every
image has positives. Known species only; the cell asserts that no novel species is used.

In [ ]:
# CELL: fusion_main
def train_fusion_pk(tag, train_species, val_species, out_dir, epochs=80, P=64, K=4,
                    lr=5e-5, wd=1e-2, patience=40):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    ck_best, ck_last, log_p = out_dir / "best.pt", out_dir / "latest.pt", out_dir / "log.json"
    by_sp = {c: np.where(CAT == c)[0] for c in sorted(train_species)}
    sp_arr = np.array(sorted(train_species))
    va_idx = np.where(np.isin(CAT, sorted(val_species)))[0]
    k_val = len(np.unique(CAT[va_idx]))
    model = TriFusion(XS_T.shape[1], XD3_T.shape[1], XD2_T.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    start, best, bad, hist = 0, -1.0, 0, []
    if ck_last.exists():
        ck = torch.load(ck_last, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        start, best, bad = ck["epoch"] + 1, ck["best"], ck["bad"]
        hist = json.load(open(log_p)) if log_p.exists() else []
        print(f"[{tag}] resuming at epoch {start + 1}, best val NMI {best:.4f}")
    for ep in range(start, epochs):
        if bad >= patience:
            break
        model.train()
        rng = np.random.default_rng(SEED + ep)
        perm = rng.permutation(sp_arr)
        tot, nb = 0.0, 0
        for i in range(0, len(perm), P):
            idx = np.concatenate([rng.choice(by_sp[c], K, replace=len(by_sp[c]) < K) for c in perm[i:i + P]])
            j = torch.as_tensor(idx, device=DEVICE)
            loss = supcon_loss(model(XS_T[j], XD3_T[j], XD2_T[j]), CAT_T[j])
            opt.zero_grad(); loss.backward(); opt.step()
            tot += float(loss); nb += 1
        f = fused_features(model, va_idx)
        nmi = normalized_mutual_info_score(CAT[va_idx], KMeans(k_val, random_state=SEED, n_init=3).fit_predict(f))
        improved = nmi > best
        if improved:
            best, bad = nmi, 0
            torch.save({"model": model.state_dict(), "best": best}, ck_best)
        else:
            bad += 1
        hist.append({"epoch": ep + 1, "loss": tot / max(nb, 1), "val_nmi": nmi, "best": best})
        json.dump(hist, open(log_p, "w"), indent=1)
        torch.save({"epoch": ep, "best": best, "bad": bad, "model": model.state_dict(), "opt": opt.state_dict()}, ck_last)
        print(f"[{tag}] ep {ep+1:3d}  loss {tot/max(nb,1):.4f}  val NMI {nmi:.4f}" + ("  *best*" if improved else ""))
    model.load_state_dict(torch.load(ck_best, map_location=DEVICE, weights_only=False)["model"])
    print(f"[{tag}] best val NMI {best:.4f}")
    return fused_features(model, np.arange(len(CAT))), model

known_ids = {c for c, r in ROLE.items() if r == "known"}
novel_ids = {c for c, r in ROLE.items() if r in ("novel_known_branch", "novel_unseen_branch")}
train_sp, val_sp = pick_val(known_ids, N_VAL_SPECIES)
assert not (novel_ids & (train_sp | val_sp)), "LEAKAGE: a novel species is in the fusion training data"
print(f"training on {len(train_sp)} known species, validating on {len(val_sp)}, {len(novel_ids)} novel held out")
F_MAIN, _ = train_fusion_pk("bioscan", train_sp, val_sp, V2_DIR / "fusion_bioscan")
np.savez(V2_DIR / "features_trifusion_bioscan.npz", X=F_MAIN, cat_ids=CAT, relpaths=REL)
json.dump({"train": sorted(train_sp), "val": sorted(val_sp)},
          open(V2_DIR / "fusion_bioscan" / "species_folds.json", "w"))
print("saved v2_bioscan/features_trifusion_bioscan.npz", F_MAIN.shape)

## Step 12 — Sanity checks

In [ ]:
# CELL: sanity
from sklearn.metrics import roc_auc_score, normalized_mutual_info_score
from sklearn.cluster import KMeans

def l2(a):
    return a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-8)

def check(X, label):
    rng = np.random.default_rng(SEED)
    roles = np.array([ROLE[c] for c in CAT])
    ref, test = [], []
    for cid in sorted({c for c, r in ROLE.items() if r == "known"}):
        m = np.where(CAT == cid)[0].copy(); rng.shuffle(m)
        k = int(round(0.7 * len(m))); ref += list(m[:k]); test += list(m[k:])
    ref, test = np.array(ref), np.array(test)
    sp = np.unique(CAT[ref])
    protos = l2(np.stack([X[ref[CAT[ref] == c]].mean(0) for c in sp]))
    novel_idx = np.where(np.isin(roles, ["novel_known_branch", "novel_unseen_branch"]))[0]
    def cluster_scores(idx):
        s = 1 - (X[idx] @ protos.T).max(1)
        return np.array([s[CAT[idx] == c].mean() for c in np.unique(CAT[idx])])
    sk, sn = cluster_scores(test), cluster_scores(novel_idx)
    auroc = roc_auc_score(np.r_[np.zeros(len(sk)), np.ones(len(sn))], np.r_[sk, sn])
    res = {"router AUROC (cluster)": auroc}
    for role in ["novel_known_branch", "novel_unseen_branch"]:
        idx = np.where(roles == role)[0]
        if len(np.unique(CAT[idx])) < 2:
            continue
        k = len(np.unique(CAT[idx]))
        pred = KMeans(k, random_state=SEED, n_init=5).fit_predict(X[idx])
        res[f"NMI {role}"] = normalized_mutual_info_score(CAT[idx], pred)
    print(f"{label:<28}" + "  ".join(f"{k} {v:.3f}" for k, v in res.items()))
    return res

concat = l2(np.concatenate([FEATS[n][0] for n in BACKBONES], axis=1))
print("Same benchmark, two representations:\n")
check(concat, "concat (training-free)")
check(F_MAIN, "trained fusion")

## Step 13 — Transfer test: the iNaturalist fusion, applied unchanged

The fusion trained on iNaturalist known species in notebook 03 is applied to the microscope images without any
retraining. This measures whether the learned representation survives a change of imaging domain.


In [ ]:
# CELL: transfer
if INAT_FUSION.exists():
    inat = TriFusion(XS_T.shape[1], XD3_T.shape[1], XD2_T.shape[1]).to(DEVICE)
    inat.load_state_dict(torch.load(INAT_FUSION, map_location=DEVICE, weights_only=False)["model"])
    F_TR = fused_features(inat, np.arange(len(CAT)))
    np.savez(V2_DIR / "features_trifusion_inat_transfer.npz", X=F_TR, cat_ids=CAT, relpaths=REL)
    check(F_TR, "iNat fusion (transfer)")
else:
    print("iNat fusion not found at", INAT_FUSION, "- skipping the transfer test")

## Step 14 — Manifest: what to copy to the analysis machine

In [ ]:
# CELL: manifest
want = ["bioscan_split.json", "items.json",
        "features_siglip2_v2.npz", "features_dinov3_v2.npz", "features_dinov2_v2.npz",
        "features_trifusion_bioscan.npz", "features_trifusion_inat_transfer.npz",
        "fusion_bioscan/best.pt", "fusion_bioscan/log.json", "fusion_bioscan/species_folds.json"]
manifest, total = [], 0
for rel in want:
    p = V2_DIR / rel
    if p.exists():
        total += p.stat().st_size; manifest.append({"file": rel, "bytes": p.stat().st_size})
        print(f"  {rel:<45}{p.stat().st_size/1e6:>9.1f} MB")
    else:
        print(f"  {rel:<45}  MISSING")
json.dump({"source": "BIOSCAN-5M (CC BY 3.0)", "species": len(SPLIT), "images": len(ITEMS),
           "roles": dict(Counter(ROLE.values())), "seed": SEED, "files": manifest},
          open(V2_DIR / "MANIFEST.json", "w"), indent=1)
print(f"\ncopy v2_bioscan/ ({total/1e9:.2f} GB, without feature_shards/) to the analysis machine.")
print("files/bioscan/ (the raw download) can stay on the GPU machine.")